In [ ]:
import numpy as np
import math
import time
import json
import warnings
warnings.filterwarnings('ignore')

from itertools import combinations, permutations
from collections import defaultdict, Counter

from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

In [ ]:
# ── Utility Functions ──

def euclidean_distance(p1, p2):
    return math.sqrt((p1[0] - p2[0])**2 + (p1[1] - p2[1])**2)

def build_distance_matrix(nodes):
    n = len(nodes)
    D = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            D[i, j] = euclidean_distance(nodes[i], nodes[j])
    return D

def route_distance(route, D):
    return sum(D[route[k], route[k + 1]] for k in range(len(route) - 1))

def infer_num_vars_from_qubo(Q):
    if not Q:
        return 0
    return 1 + max(max(i, j) for i, j in Q.keys())

def bitstring_to_array(bitstring, n):
    # Qiskit count strings are big-endian; reverse so index 0 = qubit 0
    bits = np.array([int(b) for b in bitstring[::-1]], dtype=int)
    if len(bits) < n:
        bits = np.pad(bits, (0, n - len(bits)))
    return bits[:n]

def energy_of_bitstring(bitstring_or_bits, Q, constant=0.0):
    if isinstance(bitstring_or_bits, str):
        n = infer_num_vars_from_qubo(Q)
        x = bitstring_to_array(bitstring_or_bits, n)
    else:
        x = np.asarray(bitstring_or_bits, dtype=int)

    e = constant
    for (i, j), q in Q.items():
        if i == j:
            e += q * x[i]
        else:
            e += q * x[i] * x[j]
    return float(e)

def polar_angle(depot, p):
    return math.atan2(p[1] - depot[1], p[0] - depot[0])

def angular_difference(a1, a2):
    diff = abs((a1 - a2) % (2 * math.pi))
    if diff > math.pi:
        diff = 2 * math.pi - diff
    return math.degrees(diff)

def get_k_nearest(nodes, k):
    n = len(nodes)
    D = build_distance_matrix(nodes)
    nearest = {}
    for i in range(1, n):
        dists = [(j, D[i,j]) for j in range(1, n) if i != j]
        dists.sort(key=lambda x: x[1])
        nearest[i] = [x[0] for x in dists[:min(k, len(dists))]]
    return nearest

def subset_geometry(subset, nodes):
    if len(subset) <= 1:
        return 0.0, 0.0, 0.0
    pts = [nodes[i] for i in subset]
    depot = nodes[0]
    
    max_pair = 0
    for p1 in pts:
        for p2 in pts:
            max_pair = max(max_pair, euclidean_distance(p1, p2))
            
    rads = [euclidean_distance(depot, p) for p in pts]
    rad_spread = max(rads) - min(rads)
    
    angles = [polar_angle(depot, p) for p in pts]
    max_span = 0
    for i in range(len(angles)):
        for j in range(i+1, len(angles)):
             max_span = max(max_span, angular_difference(angles[i], angles[j]))
             
    return max_pair, rad_spread, max_span


In [ ]:

def exact_subset_route_cost(subset, D):
    subset = tuple(subset)
    if len(subset) == 0: return (), 0.0
    best_order, best_cost = None, float('inf')
    for perm in permutations(subset):
        c = D[0, perm[0]]
        for k in range(len(perm) - 1):
            c += D[perm[k], perm[k + 1]]
        c += D[perm[-1], 0]
        if c < best_cost:
            best_cost = c
            best_order = perm
    return best_order, float(best_cost)

def generate_route_pool(nodes, Nv, C, config=None):
    if config is None: config = {}
    mode = config.get("route_pool_mode", "oneshot")
    use_geom = config.get("use_geometric_pruning", True)
    use_knn = config.get("use_neighbor_generator", True)
    k_knn = config.get("k_nearest", 6)
    use_cap = config.get("use_pool_cap", True)
    max_sz = config.get("max_route_pool_size", None)
    beta = config.get("beta_dist", 3.5)
    rad_thresh = config.get("radial_spread_threshold", 2.25)
    angle_thresh = config.get("angle_threshold_by_size", {2:180, 3:130, 4:95, 5:80})
    m_pair = config.get("m_pairs_per_customer", 2)
    m_high = config.get("m_higher_per_customer", 1)
    
    D = build_distance_matrix(nodes)
    customers = list(range(1, len(nodes)))
    singleton_cost = {i: 2 * D[0, i] for i in customers}
    
    nn_dists = []
    for i in customers:
        nn = [D[i,j] for j in customers if i != j]
        if nn: nn_dists.append(min(nn))
    med_nn = sorted(nn_dists)[len(nn_dists)//2] if nn_dists else 1.0
    dist_thresh = beta * med_nn
    
    knn = get_k_nearest(nodes, k_knn)
    
    all_subsets = set()
    for c in customers: all_subsets.add((c,))
    
    target_sizes = list(range(2, C + 1)) if mode == "oneshot" else [2]
    
    for r in target_sizes:
        if use_knn:
            for anchor in customers:
                neighbors = knn.get(anchor, customers)
                for p in combinations(neighbors, r-1):
                    subset = tuple(sorted((anchor,) + p))
                    all_subsets.add(subset)
        else:
            for subset in combinations(customers, r):
                all_subsets.add(tuple(sorted(subset)))
                
    route_pool = []
    for subset in all_subsets:
        if len(subset) >= 2 and use_geom:
            max_p, rad_sp, a_span = subset_geometry(subset, nodes)
            if max_p > dist_thresh: continue
            if rad_sp > rad_thresh * med_nn: continue
            if a_span > angle_thresh.get(len(subset), 360): continue
        else:
            if len(subset) >= 2:
                max_p, rad_sp, a_span = subset_geometry(subset, nodes)
            else:
                max_p = rad_sp = a_span = 0
                
        best_order, best_cost = exact_subset_route_cost(subset, D)
        savings = sum(singleton_cost[i] for i in subset) - best_cost
        density = savings / (best_cost + 1e-6)
        
        sc_density = density * config.get("score_weights", {}).get("density", 1.0)
        sc_savings = savings * config.get("score_weights", {}).get("savings", 1.0)
        sc_span    = (a_span/180) * config.get("score_weights", {}).get("span_penalty", 0.5)
        sc_pair    = (max_p/(dist_thresh+1e-6)) * config.get("score_weights", {}).get("pair_penalty", 0.5)
        
        score = sc_density + sc_savings - sc_span - sc_pair
        
        route_pool.append({
            "subset": subset, "order": list(best_order), "cost": best_cost,
            "size": len(subset), "score": score, "density_score": density,
            "savings": savings, "angular_span": a_span, "max_pairwise": max_p,
            "radial_spread": rad_sp
        })
        
    route_pool.sort(key=lambda x: (-x["score"], x["cost"], x["size"]))
    
    if use_cap:
        final_pool = [r for r in route_pool if r["size"] == 1]
        covered_pairs = {c: 0 for c in customers}
        covered_high = {c: 0 for c in customers}
        
        for r in route_pool:
            if r["size"] == 2:
                needed = any(covered_pairs[c] < m_pair for c in r["subset"])
                if needed:
                    final_pool.append(r)
                    for c in r["subset"]: covered_pairs[c] += 1
            elif r["size"] > 2:
                needed = any(covered_high[c] < m_high for c in r["subset"])
                if needed:
                    final_pool.append(r)
                    for c in r["subset"]: covered_high[c] += 1
                    
        added_subs = set(r["subset"] for r in final_pool)
        for r in route_pool:
            if max_sz and len(final_pool) >= max_sz: break
            if r["subset"] not in added_subs:
                final_pool.append(r)
                added_subs.add(r["subset"])
                
        route_pool = final_pool
        route_pool.sort(key=lambda x: (-x["score"], x["cost"], x["size"]))
        
    for idx, r in enumerate(route_pool): r["var_idx"] = idx
    return route_pool, D

def print_route_pool_summary(route_pool):
    sizes = {}
    for r in route_pool: sizes[r["size"]] = sizes.get(r["size"], 0) + 1
    print("Route-pool summary (pruned & structured):")
    for sz in sorted(sizes): print(f"  size {sz}: {sizes[sz]} routes")
    print(f"  total variables: {len(route_pool)}")



In [ ]:
# ── Phase 2: Route-Pool Selection QUBO ──

def build_subset_qubo(route_pool, n_customers, Nv, A=None, B=None):
    """
    Build the set-partitioning QUBO:

      E(y) = sum_S c_S y_S
           + A * sum_i (1 - sum_{S contains i} y_S)^2
           + B * (Nv - sum_S y_S)^2

    where each binary variable y_S selects one exact depot-to-depot route.
    """
    n_vars = len(route_pool)

    if A is None:
        A = 2.0 * max(r["cost"] for r in route_pool)
    if B is None:
        B = 2.0 * A

    Q = defaultdict(float)
    constant = A * n_customers + B * (Nv ** 2)

    # Objective: route cost
    for i, route in enumerate(route_pool):
        Q[(i, i)] += route["cost"]

    # Coverage penalty: each customer covered exactly once
    covers = defaultdict(list)
    for i, route in enumerate(route_pool):
        for cust in route["subset"]:
            covers[cust].append(i)

    for cust, idxs in covers.items():
        # (1 - sum y)^2 = 1 - sum y + 2 sum_{a<b} y_a y_b   since y^2 = y
        for i in idxs:
            Q[(i, i)] += -A
        for a in range(len(idxs)):
            for b in range(a + 1, len(idxs)):
                i, j = idxs[a], idxs[b]
                if i > j:
                    i, j = j, i
                Q[(i, j)] += 2.0 * A

    # Route-count penalty: use exactly Nv selected routes
    # (Nv - sum y)^2 = Nv^2 + (1 - 2Nv) sum y + 2 sum_{a<b} y_a y_b
    for i in range(n_vars):
        Q[(i, i)] += B * (1.0 - 2.0 * Nv)

    for i in range(n_vars):
        for j in range(i + 1, n_vars):
            Q[(i, j)] += 2.0 * B

    return dict(Q), float(constant), float(A), float(B)


def qubo_dict_to_ising(Q, n):
    """
    Convert QUBO dict to Ising coefficients using x = (1 - Z)/2.
    Returns:
      h : local Z coefficients
      J : ZZ couplings
      constant : Ising constant offset
    """
    constant = 0.0
    h = np.zeros(n)
    J = {}

    for (i, j), q in Q.items():
        if i == j:
            constant += q / 2.0
            h[i]     -= q / 2.0
        else:
            constant += q / 4.0
            h[i]     -= q / 4.0
            h[j]     -= q / 4.0
            J[(i, j)] = J.get((i, j), 0.0) + q / 4.0

    return h, J, float(constant)


def decode_route_selection(bitstring_or_bits, route_pool, Nv, n_customers):
    """
    Decode selected route-subset variables into explicit depot routes.
    Returns (routes, total_cost) if feasible, else (None, inf).
    """
    if isinstance(bitstring_or_bits, str):
        x = bitstring_to_array(bitstring_or_bits, len(route_pool))
    else:
        x = np.asarray(bitstring_or_bits, dtype=int)

    chosen = [route_pool[i] for i, bit in enumerate(x) if bit == 1]

    if len(chosen) != Nv:
        return None, float('inf')

    visited = []
    for route in chosen:
        visited.extend(route["subset"])

    if len(visited) != n_customers:
        return None, float('inf')
    if len(set(visited)) != n_customers:
        return None, float('inf')

    routes = [[0] + route["order"] + [0] for route in chosen]
    total_cost = float(sum(route["cost"] for route in chosen))
    return routes, total_cost

In [ ]:
# ── Phase 3: BF-DCQO (Notebook Scaffold) ──
#
# Important:
# This notebook uses a BF-DCQO-style iterative bias-update loop, but the
# function `build_bf_dcqo_surrogate_circuit(...)` is the ONE place you should
# swap in a more paper-faithful first-order counterdiabatic block later.
#
# Everything else in the notebook — route encoding, QUBO build, elite-sample
# bias updates, decoding, feasibility, and BBB refinement — can stay the same.


def schedule_lambda(step, total_steps):
    """
    Smooth schedule inspired by the BF/DCQO paper:
      λ(t) = sin^2( π * sin^2(π t / 2T) / 2 )
    """
    u = step / total_steps
    return float(np.sin(np.pi * (np.sin(np.pi * u / 2.0) ** 2) / 2.0) ** 2)


def build_bf_dcqo_surrogate_circuit(h, J, hb, p=3, theta_scale=1.0, theta_cutoff=0.0):
    """
    Runnable surrogate block for a BF-DCQO-style iterative solver.

    Current structure:
      - start in |+>^n
      - interleave X-mixer, Z-bias, and ZZ-problem evolutions
      - update hb externally between iterations

    Replace THIS function later with a paper-faithful CD block if desired.
    """
    n = len(h)
    qc = QuantumCircuit(n, n)
    qc.h(range(n))

    dt = 1.0 / max(1, p)

    for s in range(1, p + 1):
        lam = schedule_lambda(s, p)

        # Mixer
        mixer_angle = 2.0 * dt * (1.0 - lam)
        for q in range(n):
            qc.rx(mixer_angle, q)

        # Local fields: problem + learned bias
        for q in range(n):
            theta = 2.0 * dt * (lam * h[q] + hb[q])
            if abs(theta) > theta_cutoff:
                qc.rz(theta_scale * theta, q)

        # Pairwise ZZ terms
        for (i, j), coef in J.items():
            theta = 2.0 * dt * lam * coef
            if abs(theta) > theta_cutoff:
                qc.rzz(theta_scale * theta, i, j)

    qc.measure(range(n), range(n))
    return qc


def sample_counts(circuit, shots=1024, optimization_level=1):
    backend = AerSimulator()
    tcirc = transpile(circuit, backend=backend, optimization_level=optimization_level)
    result = backend.run(tcirc, shots=shots).result()
    counts = result.get_counts()

    gate_count = sum(tcirc.count_ops().values())
    n_qubits = tcirc.num_qubits
    return counts, gate_count, n_qubits


def elite_shots(counts, Q, constant, alpha=0.02):
    """
    Keep only the lowest-energy alpha-fraction of measured shots.
    """
    expanded = []
    for bitstring, cnt in counts.items():
        e = energy_of_bitstring(bitstring, Q, constant)
        expanded.extend([(bitstring, e)] * cnt)

    expanded.sort(key=lambda t: t[1])
    keep = max(1, int(math.ceil(alpha * len(expanded))))
    return expanded[:keep]


def compute_bias_update(counts, Q, constant, n, alpha=0.02, mode="unsigned", kappa=5.0):
    """
    Bias update convention used in this notebook:
      - unsigned iterations:
          magnitude from elite-sample <Z>,
          direction from best elite bitstring
      - final signed iteration:
          hb = kappa * elite-sample <Z>
    """
    elite = elite_shots(counts, Q, constant, alpha=alpha)

    z_sum = np.zeros(n)
    best_bits = None
    best_energy = float('inf')

    for bitstring, energy in elite:
        x = bitstring_to_array(bitstring, n)
        z = np.where(x == 0, 1.0, -1.0)   # Z expectation of computational basis state
        z_sum += z

        if energy < best_energy:
            best_energy = energy
            best_bits = x.copy()

    mags = z_sum / max(1, len(elite))

    if best_bits is None:
        return np.zeros(n), None, float('inf')

    if mode == "unsigned":
        direction = np.where(best_bits == 0, 1.0, -1.0)
        hb = direction * np.abs(mags)
    else:
        hb = kappa * mags

    return hb, best_bits, float(best_energy)


def run_bf_dcqo_generic_qubo(
    Q,
    constant,
    p=3,
    bias_iters=6,
    shots=1024,
    alpha=0.02,
    theta_scale=1.0,
    theta_cutoff=0.0,
    kappa_final=5.0,
):
    """
    Generic iterative BF-DCQO-style runner for any QUBO dict.
    Returns raw sampled distributions + iteration metadata.
    """
    n = infer_num_vars_from_qubo(Q)
    h, J, ising_constant = qubo_dict_to_ising(Q, n)

    hb = np.zeros(n)
    history = []
    iteration_counts = []

    best_energy = float('inf')
    best_bits = None
    total_gates = 0
    total_time = 0.0
    max_qubits = n

    for it in range(bias_iters):
        mode = "signed" if it == bias_iters - 1 else "unsigned"

        t0 = time.time()
        qc = build_bf_dcqo_surrogate_circuit(
            h,
            J,
            hb,
            p=p,
            theta_scale=theta_scale,
            theta_cutoff=theta_cutoff,
        )
        counts, gate_count, n_qubits = sample_counts(qc, shots=shots)
        elapsed = time.time() - t0

        iteration_counts.append(counts)
        total_gates += gate_count
        total_time += elapsed
        max_qubits = max(max_qubits, n_qubits)

        # Best raw-energy sample from this iteration
        iter_best_energy = float('inf')
        iter_best_bits = None
        for bitstring in counts:
            e = energy_of_bitstring(bitstring, Q, constant)
            if e < iter_best_energy:
                iter_best_energy = e
                iter_best_bits = bitstring_to_array(bitstring, n)

        if iter_best_energy < best_energy:
            best_energy = iter_best_energy
            best_bits = iter_best_bits.copy()

        hb, elite_best_bits, elite_best_energy = compute_bias_update(
            counts,
            Q,
            constant,
            n,
            alpha=alpha,
            mode=mode,
            kappa=kappa_final,
        )

        history.append({
            "iter": it + 1,
            "mode": mode,
            "raw_best_energy": float(iter_best_energy),
            "elite_best_energy": float(elite_best_energy),
            "gate_count": int(gate_count),
            "shots": int(shots),
        })

    return {
        "best_bits": best_bits,
        "best_energy": float(best_energy),
        "history": history,
        "iteration_counts": iteration_counts,
        "last_counts": iteration_counts[-1] if iteration_counts else {},
        "total_gates": int(total_gates),
        "total_time": float(total_time),
        "max_qubits": int(max_qubits),
    }


def run_bf_dcqo_subset_solver(
    route_pool,
    Q,
    constant,
    Nv,
    n_customers,
    p=3,
    bias_iters=6,
    shots=1024,
    alpha=0.02,
    theta_scale=1.0,
    theta_cutoff=0.0,
    kappa_final=5.0,
):
    """
    Route-subset wrapper around the generic BF-DCQO runner.
    """
    base = run_bf_dcqo_generic_qubo(
        Q=Q,
        constant=constant,
        p=p,
        bias_iters=bias_iters,
        shots=shots,
        alpha=alpha,
        theta_scale=theta_scale,
        theta_cutoff=theta_cutoff,
        kappa_final=kappa_final,
    )

    best_routes = None
    best_cost = float('inf')
    best_feasible_bits = None
    total_feasible_samples = 0
    total_samples = 0

    for counts in base["iteration_counts"]:
        for bitstring, cnt in counts.items():
            total_samples += cnt
            routes, cost = decode_route_selection(bitstring, route_pool, Nv, n_customers)
            if routes is not None:
                total_feasible_samples += cnt
                if cost < best_cost:
                    best_cost = cost
                    best_routes = routes
                    best_feasible_bits = bitstring_to_array(bitstring, len(route_pool))

    feasible_rate = total_feasible_samples / max(1, total_samples)

    return {
        **base,
        "routes": best_routes,
        "cost": float(best_cost),
        "best_feasible_bits": best_feasible_bits,
        "feasible_rate": float(feasible_rate),
    }

In [ ]:
# ── Phase 4: BBB-DCQO Selective Refinement ──
#
# Idea:
#   1. Run BF-DCQO first
#   2. Compute route-variable marginals from the final sample distribution
#   3. Fix obvious 0/1 variables
#   4. Branch only on the most uncertain variables
#   5. Solve each reduced branch with a shallower BF-DCQO run
#   6. Keep the best feasible full solution


def compute_marginals(counts, n):
    probs = np.zeros(n)
    total = sum(counts.values())

    for bitstring, cnt in counts.items():
        x = bitstring_to_array(bitstring, n)
        probs += cnt * x

    return probs / max(1, total)


def reduce_qubo_with_fixed(Q, constant, n, fixed):
    """
    Reduce QUBO by substituting fixed binary variables.
    Returns:
      Q_red, constant_red, free_vars
    """
    free_vars = [i for i in range(n) if i not in fixed]
    new_pos = {old: new for new, old in enumerate(free_vars)}

    Q_red = defaultdict(float)
    constant_red = float(constant)

    for (i, j), q in Q.items():
        vi = fixed.get(i, None)
        vj = fixed.get(j, None)

        if i == j:
            if vi is not None:
                constant_red += q * vi
            else:
                Q_red[(new_pos[i], new_pos[i])] += q
            continue

        # Off-diagonal term q * x_i * x_j
        if vi is not None and vj is not None:
            constant_red += q * vi * vj
        elif vi is not None and vj is None:
            Q_red[(new_pos[j], new_pos[j])] += q * vi
        elif vi is None and vj is not None:
            Q_red[(new_pos[i], new_pos[i])] += q * vj
        else:
            a, b = new_pos[i], new_pos[j]
            if a > b:
                a, b = b, a
            Q_red[(a, b)] += q

    return dict(Q_red), float(constant_red), free_vars


def expand_reduced_bits(reduced_bits, free_vars, fixed, n):
    full = np.zeros(n, dtype=int)
    for idx, val in fixed.items():
        full[idx] = val
    for pos, old_idx in enumerate(free_vars):
        full[old_idx] = reduced_bits[pos]
    return full


def conflicts_if_set_one(var_idx, fixed, route_pool):
    """
    Prune branch if setting y_var=1 would overlap with already-fixed y=1 routes.
    """
    candidate = set(route_pool[var_idx]["subset"])
    for idx, val in fixed.items():
        if val == 1:
            chosen = set(route_pool[idx]["subset"])
            if candidate & chosen:
                return True
    return False


def run_bbb_dcqo_refinement(
    bf_result,
    route_pool,
    Q,
    constant,
    Nv,
    n_customers,
    p_small=2,
    bias_iters_small=3,
    shots_small=512,
    alpha=0.02,
    high_fix=0.95,
    low_fix=0.05,
    max_branch_vars=6,
    max_nodes=16,
):
    """
    Selective refinement only.
    This is NOT the primary solver. It runs after BF-DCQO.
    """
    n = len(route_pool)
    counts = bf_result["last_counts"]

    if not counts:
        return {
            "routes": bf_result["routes"],
            "cost": bf_result["cost"],
            "source": "BF-DCQO (no BBB counts available)",
            "branches_tried": 0,
        }

    marginals = compute_marginals(counts, n)

    # 1) Fix obvious zeros
    fixed_base = {i: 0 for i, p in enumerate(marginals) if p <= low_fix}

    # 2) Fix obvious ones if non-conflicting
    high_candidates = [i for i, p in enumerate(marginals) if p >= high_fix]
    high_candidates.sort(key=lambda i: -marginals[i])

    for idx in high_candidates:
        if idx in fixed_base:
            continue
        if not conflicts_if_set_one(idx, fixed_base, route_pool):
            fixed_base[idx] = 1

    # 3) Branch only on most uncertain variables
    uncertain = [i for i in range(n) if i not in fixed_base]
    uncertain.sort(key=lambda i: abs(marginals[i] - 0.5))
    uncertain = uncertain[:max_branch_vars]

    best_routes = bf_result["routes"]
    best_cost = bf_result["cost"]
    branches_tried = 0

    stack = [fixed_base]

    while stack and branches_tried < max_nodes:
        fixed = stack.pop()

        # next branching variable
        next_var = None
        for idx in uncertain:
            if idx not in fixed:
                next_var = idx
                break

        # If nothing left to branch on, solve this reduced QUBO
        if next_var is None:
            Q_red, constant_red, free_vars = reduce_qubo_with_fixed(Q, constant, n, fixed)

            if len(free_vars) == 0:
                full_bits = np.array([fixed.get(i, 0) for i in range(n)], dtype=int)
                routes, cost = decode_route_selection(full_bits, route_pool, Nv, n_customers)
                if routes is not None and cost < best_cost:
                    best_routes, best_cost = routes, cost
                branches_tried += 1
                continue

            reduced = run_bf_dcqo_generic_qubo(
                Q=Q_red,
                constant=constant_red,
                p=p_small,
                bias_iters=bias_iters_small,
                shots=shots_small,
                alpha=alpha,
                theta_scale=1.0,
                theta_cutoff=0.0,
                kappa_final=5.0,
            )

            # decode best feasible full solution from all reduced-branch samples
            for counts_red in reduced["iteration_counts"]:
                for bitstring in counts_red:
                    reduced_bits = bitstring_to_array(bitstring, len(free_vars))
                    full_bits = expand_reduced_bits(reduced_bits, free_vars, fixed, n)
                    routes, cost = decode_route_selection(full_bits, route_pool, Nv, n_customers)

                    if routes is not None and cost < best_cost:
                        best_routes, best_cost = routes, cost

            branches_tried += 1
            continue

        # Branch order: try more likely value first
        p = marginals[next_var]
        preferred = 1 if p >= 0.5 else 0
        other = 1 - preferred

        # Preferred branch
        fixed_pref = dict(fixed)
        if preferred == 1 and conflicts_if_set_one(next_var, fixed_pref, route_pool):
            pass
        else:
            fixed_pref[next_var] = preferred
            stack.append(fixed_pref)

        # Alternate branch
        fixed_alt = dict(fixed)
        if other == 1 and conflicts_if_set_one(next_var, fixed_alt, route_pool):
            pass
        else:
            fixed_alt[next_var] = other
            stack.append(fixed_alt)

    source = "BBB-DCQO refinement" if best_cost < bf_result["cost"] else "BF-DCQO"
    return {
        "routes": best_routes,
        "cost": float(best_cost),
        "source": source,
        "branches_tried": int(branches_tried),
    }

In [ ]:
# ── Instance Library ──

instances = {
    1: {"Nv": 2, "C": 5, "nodes": [(0,0), (-2,2), (-5,8), (2,3)]},
    2: {"Nv": 2, "C": 2, "nodes": [(0,0), (-2,2), (-5,8), (2,3)]},
    3: {"Nv": 3, "C": 2, "nodes": [(0,0), (-2,2), (-5,8), (2,3), (5,7), (2,4), (2,-3)]},
    4: {"Nv": 4, "C": 3, "nodes": [(0,0), (-2,2), (-5,8), (6,3), (4,4), (3,2),
                                     (0,2), (-2,3), (-4,3), (2,3), (2,7), (-2,5), (-1,4)]},
}


# ── Configuration ──
read_from_file = False
selected_file_id = "instance_1"   # used when read_from_file = True
selected_id = 4                   # used when read_from_file = False

route_pool_limit = None           # None = full pool; e.g. 120 or 60 for compressed pool
use_bbb_refinement = True

# BF-DCQO params
p = 3
bias_iters = 6
shots = 1024
alpha = 0.02

# BBB refinement params
p_small = 2
bias_iters_small = 3
shots_small = 512


if read_from_file:
    with open("instances_2.json", "r") as f:
        all_instances = json.load(f)

    config = next((inst for inst in all_instances if inst["instance_id"] == selected_file_id), None)
    assert config is not None, f"Instance '{selected_file_id}' not found"

    Nv = config["Nv"]
    C = config["C"]

    raw_nodes = [(c["x"], c["y"]) for c in config["customers"]]
    nodes = raw_nodes if raw_nodes[0] == (0, 0) else [(0, 0)] + raw_nodes
    selected_id = selected_file_id
else:
    config = instances[selected_id]
    Nv = config["Nv"]
    C = config["C"]
    nodes = config["nodes"]

print(f"Loaded Instance {selected_id}: {len(nodes)-1} customers, {Nv} vehicles, capacity {C}")

config = {
    "route_pool_mode": "staged",
    "use_geometric_pruning": True,
    "use_neighbor_generator": True,
    "use_pool_cap": True,
    "k_nearest": 6,
    "beta_dist": 3.5,
    "radial_spread_threshold": 2.25,
    "angle_threshold_by_size": {2: 180.0, 3: 130.0, 4: 95.0, 5: 80.0},
    "max_route_pool_size": None,
    "m_pairs_per_customer": 2,
    "m_higher_per_customer": 1,
    "score_weights": {
        "density": 1.0,
        "savings": 1.0,
        "span_penalty": 0.5,
        "pair_penalty": 0.5
    }
}


In [ ]:
# ── Main Run ──

# 1. Generate exact route pool
route_pool, D = generate_route_pool(nodes, Nv, C, top_k=route_pool_limit)
print_route_pool_summary(route_pool)

# 2. Build QUBO
Q, constant, A, B = build_subset_qubo(
    route_pool=route_pool,
    n_customers=len(nodes) - 1,
    Nv=Nv,
)

print(f"\nPenalty coefficients:")
print(f"  A = {A:.4f}")
print(f"  B = {B:.4f}")

# 3. BF-DCQO
print(f"\n{'='*50}")
print("BF-DCQO")
print(f"{'='*50}")

bf_result = run_bf_dcqo_subset_solver(
    route_pool=route_pool,
    Q=Q,
    constant=constant,
    Nv=Nv,
    n_customers=len(nodes) - 1,
    p=p,
    bias_iters=bias_iters,
    shots=shots,
    alpha=alpha,
    theta_scale=1.0,
    theta_cutoff=0.0,
    kappa_final=5.0,
)

for row in bf_result["history"]:
    print(
        f"  iter {row['iter']:>2} | "
        f"mode={row['mode']:<8} | "
        f"raw_best_E={row['raw_best_energy']:>10.3f} | "
        f"elite_best_E={row['elite_best_energy']:>10.3f} | "
        f"gates={row['gate_count']}"
    )

print(f"\nBF best feasible cost: {bf_result['cost']:.4f}")
print(f"BF feasible sample rate: {bf_result['feasible_rate']:.4f}")
print(f"BF qubits: {bf_result['max_qubits']}")
print(f"BF total gates: {bf_result['total_gates']}")
print(f"BF total time: {bf_result['total_time']:.2f}s")

candidate_routes = bf_result["routes"]
candidate_cost = bf_result["cost"]
candidate_source = "BF-DCQO"

# 4. Optional BBB refinement
if use_bbb_refinement:
    print(f"\n{'='*50}")
    print("BBB-DCQO REFINEMENT")
    print(f"{'='*50}")

    bbb_result = run_bbb_dcqo_refinement(
        bf_result=bf_result,
        route_pool=route_pool,
        Q=Q,
        constant=constant,
        Nv=Nv,
        n_customers=len(nodes) - 1,
        p_small=p_small,
        bias_iters_small=bias_iters_small,
        shots_small=shots_small,
        alpha=alpha,
        high_fix=0.95,
        low_fix=0.05,
        max_branch_vars=6,
        max_nodes=16,
    )

    print(f"BBB branches tried: {bbb_result['branches_tried']}")
    print(f"Selected source: {bbb_result['source']}")
    print(f"Refined best cost: {bbb_result['cost']:.4f}")

    if bbb_result["routes"] is not None and bbb_result["cost"] < candidate_cost:
        candidate_routes = bbb_result["routes"]
        candidate_cost = bbb_result["cost"]
        candidate_source = bbb_result["source"]

# 5. Final chosen solution
routes = candidate_routes
total_distance = candidate_cost

print(f"\n{'='*50}")
print(f"FINAL SOLUTION ({candidate_source})")
print(f"{'='*50}")

if routes is None:
    print("No feasible solution was decoded.")
else:
    for i, route in enumerate(routes):
        rd = route_distance(route, D)
        print(f"  r{i+1}: {', '.join(map(str, route))}  |  Distance: {rd:.2f}")

    print(f"\nTotal Distance: {total_distance:.4f}")

In [ ]:
# ── Feasibility Check (same style as your QAOA notebook) ──

print("=" * 50)
print("FEASIBILITY CHECK")
print("=" * 50)

all_customers = set(range(1, len(nodes)))
visited = []
feasible = True

in_degree = {i: 0 for i in range(len(nodes))}
out_degree = {i: 0 for i in range(len(nodes))}

if routes is None:
    feasible = False
    print("  ✗ No decoded routes available")
else:
    for i, route in enumerate(routes):
        # 1. Route starts and ends at depot
        if route[0] != 0 or route[-1] != 0:
            print(f"  ✗ Route {i+1} does not start/end at depot: {route}")
            feasible = False
        else:
            print(f"  ✓ Route {i+1} starts and ends at depot")

        # 2. Capacity constraint
        customers_in_route = [c for c in route if c != 0]
        if len(customers_in_route) > C:
            print(f"  ✗ Route {i+1} exceeds capacity: {len(customers_in_route)} > {C}")
            feasible = False
        else:
            print(f"  ✓ Route {i+1} capacity OK ({len(customers_in_route)}/{C})")

        # 3. No repeated customers within route
        if len(customers_in_route) != len(set(customers_in_route)):
            print(f"  ✗ Route {i+1} has repeated customers: {route}")
            feasible = False
        else:
            print(f"  ✓ Route {i+1} no internal repeats")

        # Flow bookkeeping
        for k in range(len(route) - 1):
            out_degree[route[k]] += 1
            in_degree[route[k + 1]] += 1

        visited.extend(customers_in_route)

    # 4. Every customer visited exactly once
    visited_set = set(visited)
    missing = all_customers - visited_set
    duplicates = [c for c in visited if visited.count(c) > 1]

    if missing:
        print(f"  ✗ Missing customers: {missing}")
        feasible = False
    else:
        print(f"  ✓ All {len(all_customers)} customers visited")

    if duplicates:
        print(f"  ✗ Duplicate visits: {set(duplicates)}")
        feasible = False
    else:
        print(f"  ✓ No duplicate visits")

    # 5. Number of vehicles
    if len(routes) > Nv:
        print(f"  ✗ Too many vehicles: {len(routes)} > {Nv}")
        feasible = False
    else:
        print(f"  ✓ Vehicles used: {len(routes)}/{Nv}")

    # 6. Flow conservation
    print(f"\n{'─'*50}")
    print("FLOW CONSERVATION")
    print(f"{'─'*50}")

    depot_out_ok = out_degree[0] == len(routes)
    depot_in_ok = in_degree[0] == len(routes)

    if depot_out_ok and depot_in_ok:
        print(f"  ✓ Depot flow balanced: {out_degree[0]} out, {in_degree[0]} in")
    else:
        print(f"  ✗ Depot flow imbalanced: {out_degree[0]} out, {in_degree[0]} in")
        feasible = False

    flow_violations = []
    for c in all_customers:
        if in_degree[c] != 1 or out_degree[c] != 1:
            flow_violations.append((c, in_degree[c], out_degree[c]))

    if not flow_violations:
        print(f"  ✓ All customers: in-degree = out-degree = 1")
    else:
        for c, ind, outd in flow_violations:
            print(f"  ✗ Customer {c}: in-degree={ind}, out-degree={outd}")
        feasible = False

    # 7. Distance verification
    print(f"\n{'─'*50}")
    print("DISTANCE VERIFICATION")
    print(f"{'─'*50}")

    recomputed = 0.0
    for i, route in enumerate(routes):
        rd = route_distance(route, D)
        recomputed += rd
        print(f"  → Route {i+1}: {' → '.join(map(str, route))}  dist={rd:.2f}")

    if abs(recomputed - total_distance) < 1e-6:
        print(f"  ✓ Total distance verified: {recomputed:.4f}")
    else:
        print(f"  ✗ Distance mismatch: reported={total_distance:.4f}, recomputed={recomputed:.4f}")
        feasible = False

print(f"\n{'='*50}")
print(f"{'✓ SOLUTION FEASIBLE' if feasible else '✗ SOLUTION INFEASIBLE'}")
print(f"{'='*50}")

In [ ]:
# ── Optional: Run All Instances with the New Solver ──

summary_rows = []

for inst_id, config in instances.items():
    Nv_inst = config["Nv"]
    C_inst = config["C"]
    nodes_inst = config["nodes"]
    n_cust = len(nodes_inst) - 1

    print(f"\n{'█'*60}")
    print(f"  INSTANCE {inst_id}: {n_cust} customers, {Nv_inst} vehicles, capacity {C_inst}")
    print(f"{'█'*60}")

    route_pool_inst, D_inst = generate_route_pool(nodes_inst, Nv_inst, C_inst, top_k=None)
    Q_inst, constant_inst, A_inst, B_inst = build_subset_qubo(
        route_pool_inst,
        n_customers=n_cust,
        Nv=Nv_inst,
    )

    bf_inst = run_bf_dcqo_subset_solver(
        route_pool=route_pool_inst,
        Q=Q_inst,
        constant=constant_inst,
        Nv=Nv_inst,
        n_customers=n_cust,
        p=3,
        bias_iters=6,
        shots=1024,
        alpha=0.02,
    )

    best_routes = bf_inst["routes"]
    best_cost = bf_inst["cost"]
    source = "BF-DCQO"

    bbb_inst = run_bbb_dcqo_refinement(
        bf_result=bf_inst,
        route_pool=route_pool_inst,
        Q=Q_inst,
        constant=constant_inst,
        Nv=Nv_inst,
        n_customers=n_cust,
        p_small=2,
        bias_iters_small=3,
        shots_small=512,
        alpha=0.02,
        high_fix=0.95,
        low_fix=0.05,
        max_branch_vars=6,
        max_nodes=16,
    )

    if bbb_inst["routes"] is not None and bbb_inst["cost"] < best_cost:
        best_routes = bbb_inst["routes"]
        best_cost = bbb_inst["cost"]
        source = "BBB-DCQO"

    print(f"  Selected source: {source}")
    if best_routes is not None:
        for i, route in enumerate(best_routes):
            print(f"  r{i+1}: {' → '.join(map(str, route))}  |  dist={route_distance(route, D_inst):.2f}")
        print(f"  Total distance: {best_cost:.4f}")
    else:
        print("  No feasible solution decoded")

    summary_rows.append((
        inst_id,
        n_cust,
        Nv_inst,
        C_inst,
        best_cost,
        bf_inst["max_qubits"],
        bf_inst["total_gates"],
        bf_inst["total_time"],
        source,
        best_routes is not None,
    ))

print(f"\n\n{'='*105}")
print(f"{'SUMMARY':^105}")
print(f"{'='*105}")
print(f"{'Inst':>5} {'Cust':>5} {'Veh':>4} {'Cap':>4} {'Distance':>12} {'Qubits':>7} {'Gates':>8} {'Time(s)':>9} {'Source':>14} {'Status':>12}")
print(f"{'─'*105}")
for inst_id, nc, nv, cap, dist, qb, gt, tm, src, ok in summary_rows:
    tag = "✓ FEASIBLE" if ok else "✗ INFEAS."
    print(f"{inst_id:>5} {nc:>5} {nv:>4} {cap:>4} {dist:>12.4f} {qb:>7} {gt:>8} {tm:>9.2f} {src:>14} {tag:>12}")
print(f"{'='*105}")